In [2]:
pip install --upgrade pip && pip install "numpy==1.26.4" snntorch h5py scikit-learn matplotlib

  Using cached pip-26.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.0
    Uninstalling pip-25.0:
      Successfully uninstalled pip-25.0
  Using cached snntorch-0.9.4-py2.py3-none-any.whl.metadata (15 kB)
Using cached snntorch-0.9.4-py2.py3-none-any.whl (125 kB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
# gen_fc_from_dcll.py
import torch
import numpy as np

import pytorch_libdcll as dcll

# -------------------------
# Reproducibility
# -------------------------
torch.manual_seed(0)
np.random.seed(0)

# -------------------------
# Small FC dimensions
# -------------------------
IN_CH = 8
OUT_CH = 4
BATCH = 1

# -------------------------
# Build DCLL dense module directly
# -------------------------
layer = dcll.CLLDenseModule(
    in_channels=IN_CH,
    out_channels=OUT_CH,
    bias=True,
    alpha=0.0,
    alphas=0.0,
    spiking=True
)

# The source hardcodes device='cuda', so force CPU-safe tensors manually
# instead of relying on init_state()
layer = layer.cpu()
layer.alpha = torch.nn.Parameter(torch.tensor([0.0]), requires_grad=False)
layer.tau_m__dt = torch.nn.Parameter(torch.tensor([1.0]), requires_grad=False)
layer.alphas = torch.nn.Parameter(torch.tensor([0.0]), requires_grad=False)
layer.tau_s__dt = torch.nn.Parameter(torch.tensor([1.0]), requires_grad=False)

layer.state = layer.NeuronState(
    eps0=torch.zeros(BATCH, IN_CH),
    eps1=torch.zeros(BATCH, IN_CH)
)

# -------------------------
# Random integer input / weights / bias
# -------------------------
x = torch.randint(low=-2, high=3, size=(BATCH, IN_CH)).float()
W = torch.randint(low=-2, high=3, size=(OUT_CH, IN_CH)).float()
b = torch.randint(low=-2, high=3, size=(OUT_CH,)).float()

with torch.no_grad():
    layer.weight.copy_(W)
    layer.bias.copy_(b)

# -------------------------
# Run direct DCLL lib forward
# -------------------------
spk, pv, vmem = layer(x)

eps0 = layer.state.eps0.clone()
eps1 = layer.state.eps1.clone()

# -------------------------
# Print reference values
# -------------------------
print("x =", x.int().tolist())
print("W =", W.int().tolist())
print("b =", b.int().tolist())
print("eps0 =", eps0.tolist())
print("eps1 =", eps1.tolist())
print("vmem =", vmem.tolist())
print("spk =", spk.int().tolist())

# -------------------------
# Export compact header for HLS TB
# -------------------------
with open("fc_dcll_ref.h", "w") as f:
    f.write("#ifndef FC_DCLL_REF_H\n")
    f.write("#define FC_DCLL_REF_H\n\n")

    # input
    f.write(f"static const int FC_IN[{IN_CH}] = {{")
    f.write(", ".join(str(int(v)) for v in x[0].tolist()))
    f.write("};\n\n")

    # weights
    f.write(f"static const int FC_W[{OUT_CH}][{IN_CH}] = {{\n")
    for r in W.int().tolist():
        f.write("    {" + ", ".join(str(int(v)) for v in r) + "},\n")
    f.write("};\n\n")

    # bias
    f.write(f"static const int FC_B[{OUT_CH}] = {{")
    f.write(", ".join(str(int(v)) for v in b.tolist()))
    f.write("};\n\n")

    # expected eps0 / eps1 / vmem / spike
    f.write(f"static const int FC_EXPECT_EPS0[{IN_CH}] = {{")
    f.write(", ".join(str(int(round(v))) for v in eps0[0].tolist()))
    f.write("};\n\n")

    f.write(f"static const int FC_EXPECT_EPS1[{IN_CH}] = {{")
    f.write(", ".join(str(int(round(v))) for v in eps1[0].tolist()))
    f.write("};\n\n")

    f.write(f"static const int FC_EXPECT_VMEM[{OUT_CH}] = {{")
    f.write(", ".join(str(int(round(v))) for v in vmem[0].tolist()))
    f.write("};\n\n")

    f.write(f"static const int FC_EXPECT_SPK[{OUT_CH}] = {{")
    f.write(", ".join(str(int(v)) for v in spk[0].int().tolist()))
    f.write("};\n\n")

    f.write("#endif\n")

x = [[0, 2, -1, 1, 1, 1, -1, 2]]
W = [[-1, 0, 1, -2, 0, -1, -2, 2], [1, -1, -1, -2, 1, -1, -1, 0], [2, -1, 1, 2, -2, -2, 1, 0], [1, 0, 0, -2, -2, -2, 1, -1]]
b = [2, -1, -1, -2]
eps0 = [[0.0, 2.0, -1.0, 1.0, 1.0, 1.0, -1.0, 2.0]]
eps1 = [[0.0, 2.0, -1.0, 1.0, 1.0, 1.0, -1.0, 2.0]]
vmem = [[4.0, -3.0, -7.0, -11.0]]
spk = [[1, 0, 0, 0]]


In [3]:
# gen_conv1x1_case.py
import torch
import numpy as np
import pytorch_libdcll as dcll

torch.manual_seed(0)
np.random.seed(0)

# -------------------------
# Tiny test:
# batch=1, in_ch=1, out_ch=1, H=W=1, kernel=1
# -------------------------
B = 1
IC = 1
OC = 1
H = 1
W = 1
K = 1

layer = dcll.ContinuousConv2D(
    in_channels=IC,
    out_channels=OC,
    kernel_size=K,
    stride=1,
    padding=0,
    dilation=1,
    groups=1,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
)

# Force CPU-safe behavior
layer = layer.cpu()
layer.alpha = torch.tensor([0.0])
layer.tau_m__dt = torch.tensor([1.0])
layer.alphas = torch.tensor([0.0])
layer.tau_s__dt = torch.tensor([1.0])

# Zero initial state
layer.state = layer.NeuronState(
    eps0=torch.zeros((B, OC, H, W)),
    eps1=torch.zeros((B, OC, H, W))
)

# -------------------------
# Fixed example values
# -------------------------
# Input shape: [B, IC, H, W]
x = torch.tensor([[[[2.0]]]])

# Weight shape: [OC, IC, K, K]
w = torch.tensor([[[[3.0]]]])

with torch.no_grad():
    layer.weight.copy_(w)

# -------------------------
# Run Python DCLL conv
# -------------------------
spk, pv, pvmem = layer(x, save=False)

eps0 = layer.state.eps0.clone()
eps1 = layer.state.eps1.clone()

print("x =", x.tolist())
print("w =", w.tolist())
print("eps0 =", eps0.tolist())
print("eps1 =", eps1.tolist())
print("pvmem =", pvmem.tolist())
print("spk =", spk.int().tolist())

# For this testcase:
# conv2d(x,w)=6
# eps0=6, eps1=6, spk=1

with open("conv1x1_ref.h", "w") as f:
    f.write("#ifndef CONV1X1_REF_H\n")
    f.write("#define CONV1X1_REF_H\n\n")
    f.write("static const int IN_VAL = 2;\n")
    f.write("static const int W_VAL = 3;\n")
    f.write("static const int EXPECT_EPS0 = 6;\n")
    f.write("static const int EXPECT_EPS1 = 6;\n")
    f.write("static const int EXPECT_PVMEM = 6;\n")
    f.write("static const int EXPECT_SPK = 1;\n\n")
    f.write("#endif\n")

Sigmoid()
Continuous 2D
x = [[[[2.0]]]]
w = [[[[3.0]]]]
eps0 = [[[[6.0]]]]
eps1 = [[[[6.0]]]]
pvmem = [[[[6.0]]]]
spk = [[[[1]]]]


In [2]:
import torch
import numpy as np
import pytorch_libdcll as dcll

torch.manual_seed(0)
np.random.seed(0)

B = 1
IC = 1
OC = 1
H = 2
W = 2
K = 1

layer = dcll.ContinuousConv2D(
    in_channels=IC,
    out_channels=OC,
    kernel_size=K,
    stride=1,
    padding=0,
    dilation=1,
    groups=1,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
)

layer = layer.cpu()
layer.alpha = torch.tensor([0.0])
layer.tau_m__dt = torch.tensor([1.0])
layer.alphas = torch.tensor([0.0])
layer.tau_s__dt = torch.tensor([1.0])

layer.state = layer.NeuronState(
    eps0=torch.zeros((B, OC, H, W)),
    eps1=torch.zeros((B, OC, H, W))
)

# binary input, comparable to FPGA mask-based path
x = torch.tensor([[[[1.0, 0.0],
                    [1.0, 0.0]]]])

w = torch.tensor([[[[3.0]]]])

with torch.no_grad():
    layer.weight.copy_(w)

spk, pv, pvmem = layer(x, save=False)

print("pvmem =", pvmem.tolist())
print("spk =", spk.int().tolist())

Sigmoid()
Continuous 2D
pvmem = [[[[3.0, 0.0], [3.0, 0.0]]]]
spk = [[[[1, 0], [1, 0]]]]


In [1]:
import torch
import numpy as np
import pytorch_libdcll as dcll

torch.manual_seed(0)
np.random.seed(0)

B = 1
IC = 1
OC = 1
H = 3
W = 3
K = 2

layer = dcll.ContinuousConv2D(
    in_channels=IC,
    out_channels=OC,
    kernel_size=K,
    stride=1,
    padding=0,
    dilation=1,
    groups=1,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
)

layer = layer.cpu()
layer.alpha = torch.tensor([0.0])
layer.tau_m__dt = torch.tensor([1.0])
layer.alphas = torch.tensor([0.0])
layer.tau_s__dt = torch.tensor([1.0])

# output is (H-K+1, W-K+1) = (2,2)
OH = H - K + 1
OW = W - K + 1

layer.state = layer.NeuronState(
    eps0=torch.zeros((B, OC, OH, OW)),
    eps1=torch.zeros((B, OC, OH, OW))
)

# binary input
x = torch.tensor([[[[1.0, 0.0, 1.0],
                    [0.0, 1.0, 0.0],
                    [1.0, 1.0, 0.0]]]])

# real 2x2 conv kernel
w = torch.tensor([[[[ 2.0, -1.0],
                    [ 1.0,  3.0]]]])

with torch.no_grad():
    layer.weight.copy_(w)

spk, pv, pvmem = layer(x, save=False)

print("input x =")
print(x[0, 0].tolist())

print("weight w =")
print(w[0, 0].tolist())

print("pvmem =")
print(pvmem.tolist())

print("spk =")
print(spk.int().tolist())

Sigmoid()
Continuous 2D
input x =
[[1.0, 0.0, 1.0], [0.0, 1.0, 0.0], [1.0, 1.0, 0.0]]
weight w =
[[2.0, -1.0], [1.0, 3.0]]
pvmem =
[[[[5.0, 0.0], [3.0, 3.0]]]]
spk =
[[[[1, 0], [1, 1]]]]


In [4]:
import torch
from pytorch_libdcll import CLLDenseModule

dev = torch.device("cuda")

x = torch.tensor([[1, 0, 1, 0, 1, 0, 1, 0]], dtype=torch.float32, device=dev)

W = torch.tensor([
    [ 1, 0, 1, 0, 1, 0, 1, 0],
    [ 1, 1, 1, 1, 0, 0, 0, 0],
    [-1, 0,-1, 0,-1, 0,-1, 0],
    [ 0, 0, 0, 0, 1, 1, 1, 1],
], dtype=torch.float32, device=dev)

layer = CLLDenseModule(
    in_channels=8,
    out_channels=4,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
).to(dev)

with torch.no_grad():
    layer.weight.copy_(W)

layer.init_state(batch_size=1, init_value=0)

spikes, pv, pvmem = layer(x)

print("pvmem :", pvmem.int().cpu().tolist())
print("spikes:", spikes.int().cpu().tolist())

pvmem : [[4, 2, -4, 2]]
spikes: [[1, 1, 0, 1]]


SNN CONV2D size

In [3]:
import torch
import torch.nn as nn

MAX_CONSTITUENTS = 10
NUM_CLASSES = 5

class HardFINNLIF(nn.Module):
    def __init__(self, decay=0.0, threshold=0.0):
        super().__init__()
        self.decay = decay
        self.threshold = threshold

    def forward(self, x, mem):
        mem = mem * self.decay + x
        spk = (mem > self.threshold).float()
        mem = mem * (1.0 - spk)
        return spk, mem


class ConvSNN(nn.Module):
    def __init__(self, input_dim=3, decay=0.0, threshold=0.0):
        super().__init__()

        self.conv1 = nn.Conv2d(input_dim, 32, kernel_size=(1, 1), bias=False)
        self.lif1 = HardFINNLIF(decay=decay, threshold=threshold)

        self.fc1 = nn.Linear(32 * MAX_CONSTITUENTS, 64, bias=False)
        self.lif2 = HardFINNLIF(decay=decay, threshold=threshold)

        self.fc2 = nn.Linear(64, NUM_CLASSES, bias=False)
        self.lif3 = HardFINNLIF(decay=decay, threshold=threshold)

    def forward(self, x):
        print("=== Conv2D (1x1) ===")

        conv_out = self.conv1(x)
        mem1 = torch.zeros_like(conv_out)
        spk1, mem1 = self.lif1(conv_out, mem1)

        print("input x =")
        print(x.squeeze(0).squeeze(1).int().tolist())

        print("weight w (first 4 filters) =")
        print(self.conv1.weight[:4, :, 0, 0].int().tolist())

        print("pvmem =")
        print(conv_out.squeeze(0).squeeze(1).tolist())

        print("spk =")
        print(spk1.squeeze(0).squeeze(1).int().tolist())

        x_flat = spk1.float().view(1, -1)

        print("\n=== FC1 ===")

        fc1_out = self.fc1(x_flat)
        mem2 = torch.zeros_like(fc1_out)
        spk2, mem2 = self.lif2(fc1_out, mem2)

        print("input x =")
        print(x_flat.int().tolist())

        print("weight w (first 4 neurons, first 16 inputs) =")
        print(self.fc1.weight[:4, :16].int().tolist())

        print("pvmem =")
        print(fc1_out.tolist())

        print("spk =")
        print(spk2.int().tolist())

        spk2_float = spk2.float()

        print("\n=== FC2 ===")

        fc2_out = self.fc2(spk2_float)
        mem3 = torch.zeros_like(fc2_out)
        spk3, mem3 = self.lif3(fc2_out, mem3)

        print("input x =")
        print(spk2_float.int().tolist())

        print("weight w =")
        print(self.fc2.weight.int().tolist())

        print("pvmem =")
        print(fc2_out.tolist())

        print("spk =")
        print(spk3.int().tolist())

        return spk1, spk2, spk3


# ======================
# RUN TEST
# ======================

torch.manual_seed(0)

model = ConvSNN(input_dim=3, decay=0.0, threshold=0.0)

# input shape: batch=1, channels=3, height=1, width=MAX_CONSTITUENTS
x = torch.randint(0, 2, (1, 3, 1, MAX_CONSTITUENTS)).float()

# integer-like random weights for HLS-friendly testing
with torch.no_grad():
    model.conv1.weight.copy_(torch.randint(-2, 3, model.conv1.weight.shape).float())
    model.fc1.weight.copy_(torch.randint(-2, 3, model.fc1.weight.shape).float())
    model.fc2.weight.copy_(torch.randint(-2, 3, model.fc2.weight.shape).float())

spk1, spk2, spk3 = model(x)

=== Conv2D (1x1) ===
input x =
[[0, 0, 1, 0, 0, 0, 1, 1, 0, 1], [0, 0, 1, 1, 0, 1, 0, 1, 1, 0], [1, 1, 1, 0, 0, 1, 1, 0, 0, 1]]
weight w (first 4 filters) =
[[-1, -2, 2], [-1, 1, 0], [0, 0, -1], [1, 1, 0]]
pvmem =
[[2.0, 2.0, -1.0, -2.0, 0.0, 0.0, 1.0, -3.0, -2.0, 1.0], [0.0, 0.0, 0.0, 1.0, 0.0, 1.0, -1.0, 0.0, 1.0, -1.0], [-1.0, -1.0, -1.0, 0.0, 0.0, -1.0, -1.0, 0.0, 0.0, -1.0], [0.0, 0.0, 2.0, 1.0, 0.0, 1.0, 1.0, 2.0, 1.0, 1.0], [1.0, 1.0, 0.0, -2.0, 0.0, -1.0, 2.0, -1.0, -2.0, 2.0], [2.0, 2.0, -1.0, -1.0, 0.0, 1.0, 0.0, -3.0, -1.0, 0.0], [0.0, 0.0, 3.0, 1.0, 0.0, 1.0, 2.0, 3.0, 1.0, 2.0], [-1.0, -1.0, 0.0, 1.0, 0.0, 0.0, -1.0, 1.0, 1.0, -1.0], [2.0, 2.0, 1.0, 0.0, 0.0, 2.0, 1.0, -1.0, 0.0, 1.0], [-1.0, -1.0, 1.0, 2.0, 0.0, 1.0, -1.0, 2.0, 2.0, -1.0], [1.0, 1.0, 3.0, 2.0, 0.0, 3.0, 1.0, 2.0, 2.0, 1.0], [2.0, 2.0, 5.0, 1.0, 0.0, 3.0, 4.0, 3.0, 1.0, 4.0], [0.0, 0.0, 1.0, -1.0, 0.0, -1.0, 2.0, 1.0, -1.0, 2.0], [0.0, 0.0, -1.0, 0.0, 0.0, 0.0, -1.0, -1.0, 0.0, -1.0], [-1.0, -1.0, 2.0, 2.0

In [4]:
import numpy as np

def write_cpp_array_1d(f, name, arr):
    arr = np.array(arr).astype(int).reshape(-1)
    f.write(f"static const int {name}[{len(arr)}] = {{\n")
    for i in range(0, len(arr), 16):
        f.write("  " + ", ".join(map(str, arr[i:i+16])) + ",\n")
    f.write("};\n\n")

def write_cpp_array_2d(f, name, arr):
    arr = np.array(arr).astype(int)
    rows, cols = arr.shape
    f.write(f"static const int {name}[{rows}][{cols}] = {{\n")
    for r in range(rows):
        f.write("  {" + ", ".join(map(str, arr[r])) + "},\n")
    f.write("};\n\n")

# after running:
# spk1, spk2, spk3 = model(x)

conv_in   = x.squeeze(0).squeeze(1).int().cpu().numpy()          # [3][10]
conv_w    = model.conv1.weight[:, :, 0, 0].int().cpu().numpy()   # [32][3]
conv_spk  = spk1.squeeze(0).squeeze(1).int().cpu().numpy()       # [32][10]

fc1_in    = spk1.float().view(-1).int().cpu().numpy()            # [320]
fc1_w     = model.fc1.weight.int().cpu().numpy()                 # [64][320]
fc1_spk   = spk2.view(-1).int().cpu().numpy()                    # [64]

fc2_in    = spk2.float().view(-1).int().cpu().numpy()            # [64]
fc2_w     = model.fc2.weight.int().cpu().numpy()                 # [5][64]
fc2_spk   = spk3.view(-1).int().cpu().numpy()                    # [5]

with open("conv1_ref.h", "w") as f:
    f.write("#ifndef CONV1_REF_H\n#define CONV1_REF_H\n\n")
    write_cpp_array_2d(f, "CONV_IN", conv_in)
    write_cpp_array_2d(f, "CONV_W", conv_w)
    write_cpp_array_2d(f, "CONV_EXPECT", conv_spk)
    f.write("#endif\n")

with open("fc1_ref.h", "w") as f:
    f.write("#ifndef FC1_REF_H\n#define FC1_REF_H\n\n")
    write_cpp_array_1d(f, "FC1_IN", fc1_in)
    write_cpp_array_2d(f, "FC1_W", fc1_w)
    write_cpp_array_1d(f, "FC1_EXPECT", fc1_spk)
    f.write("#endif\n")

with open("fc2_ref.h", "w") as f:
    f.write("#ifndef FC2_REF_H\n#define FC2_REF_H\n\n")
    write_cpp_array_1d(f, "FC2_IN", fc2_in)
    write_cpp_array_2d(f, "FC2_W", fc2_w)
    write_cpp_array_1d(f, "FC2_EXPECT", fc2_spk)
    f.write("#endif\n")

In [6]:
import torch
import torch.nn as nn

# ======================
# CONFIG
# ======================
BATCH = 1
INPUT_DIM = 3
MAX_CONSTITUENTS = 10
NUM_CLASSES = 5

torch.manual_seed(0)  # deterministic

# ======================
# SIMPLE HARD LIF (HLS-style)
# ======================
class HardLIF(nn.Module):
    def __init__(self, threshold=0.0):
        super().__init__()
        self.threshold = threshold

    def forward(self, x):
        spk = (x > self.threshold).int()
        return spk


# ======================
# MODEL
# ======================
class TestSNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(INPUT_DIM, 32, kernel_size=(1, 1), bias=False)
        self.fc1 = nn.Linear(32 * MAX_CONSTITUENTS, 64, bias=False)
        self.fc2 = nn.Linear(64, NUM_CLASSES, bias=False)

        self.lif = HardLIF(threshold=0.0)

    def forward(self, x):

        print("\n===== INPUT =====")
        print(x.int().tolist())

        # ---- CONV ----
        conv = self.conv1(x)

        print("\n===== CONV WEIGHT =====")
        print(self.conv1.weight[:, :, 0, 0].int().tolist())

        print("\n===== CONV OUT =====")
        print(conv.tolist())

        spk1 = self.lif(conv)

        print("\n===== CONV SPIKE =====")
        print(spk1.tolist())

        # ---- FLATTEN ----
        flat = spk1.view(x.size(0), -1)

        print("\n===== FLATTEN =====")
        print(flat.tolist())

        # ---- FC1 ----
        fc1 = self.fc1(flat.float())

        print("\n===== FC1 WEIGHT (first 4 rows, first 16 inputs) =====")
        print(self.fc1.weight[:4, :16].int().tolist())

        print("\n===== FC1 OUT =====")
        print(fc1.tolist())

        spk2 = self.lif(fc1)

        print("\n===== FC1 SPIKE =====")
        print(spk2.tolist())

        # ---- FC2 ----
        fc2 = self.fc2(spk2.float())

        print("\n===== FC2 WEIGHT =====")
        print(self.fc2.weight.int().tolist())

        print("\n===== FC2 OUT =====")
        print(fc2.tolist())

        spk3 = self.lif(fc2)

        print("\n===== FINAL SPIKE (OUTPUT) =====")
        print(spk3.tolist())

        return spk3


# ======================
# GENERATE RANDOM INPUT
# ======================
x = torch.randint(
    0, 2,
    (BATCH, INPUT_DIM, 1, MAX_CONSTITUENTS)
).float()   # ⚠️ MUST be float for Conv2d

# ======================
# INIT MODEL
# ======================
model = TestSNN()

# integer-like weights (HLS-friendly)
with torch.no_grad():
    model.conv1.weight.copy_(torch.randint(-2, 3, model.conv1.weight.shape).float())
    model.fc1.weight.copy_(torch.randint(-2, 3, model.fc1.weight.shape).float())
    model.fc2.weight.copy_(torch.randint(-2, 3, model.fc2.weight.shape).float())

# ======================
# RUN
# ======================
out = model(x)


===== INPUT =====
[[[[0, 1, 1, 0, 1, 1, 1, 1, 1, 1]], [[1, 0, 0, 1, 0, 0, 0, 0, 0, 1]], [[0, 1, 1, 0, 0, 1, 1, 1, 1, 0]]]]

===== CONV WEIGHT =====
[[-1, -2, 2], [-1, 1, 0], [0, 0, -1], [1, 1, 0], [1, -2, 1], [-2, -1, 2], [2, 1, 0], [0, 1, -1], [-1, 0, 2], [0, 2, -1], [0, 2, 1], [2, 1, 2], [2, -1, 0], [-1, 0, 0], [1, 2, -1], [2, -1, -1], [0, 2, -2], [-1, -1, 1], [2, 0, 1], [-2, 0, 1], [-2, 0, 2], [-1, 1, 1], [-1, -2, 2], [2, 1, 1], [1, -2, 1], [0, -2, 2], [1, 1, 1], [0, -2, -1], [2, -1, 0], [0, 1, 0], [2, 1, 0], [1, 2, -2]]

===== CONV OUT =====
[[[[-2.0, 1.0, 1.0, -2.0, -1.0, 1.0, 1.0, 1.0, 1.0, -3.0]], [[1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 0.0]], [[0.0, -1.0, -1.0, 0.0, 0.0, -1.0, -1.0, -1.0, -1.0, 0.0]], [[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 2.0]], [[-2.0, 2.0, 2.0, -2.0, 1.0, 2.0, 2.0, 2.0, 2.0, -1.0]], [[-1.0, 0.0, 0.0, -1.0, -2.0, 0.0, 0.0, 0.0, 0.0, -3.0]], [[1.0, 2.0, 2.0, 1.0, 2.0, 2.0, 2.0, 2.0, 2.0, 3.0]], [[1.0, -1.0, -1.0, 1.0, 0.0, -1.0, -1.0, -1

In [ ]:
dcll

In [7]:
import torch
import numpy as np
import pytorch_libdcll as dcll
from pytorch_libdcll import CLLDenseModule

# ======================
# CONFIG
# ======================
BATCH = 1
INPUT_DIM = 3
MAX_CONSTITUENTS = 10
NUM_CLASSES = 5

torch.manual_seed(0)
np.random.seed(0)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

# ======================
# DCLL MODEL
# conv1: 3 -> 32, 1x1
# fc1: 320 -> 64
# fc2: 64 -> 5
# ======================

conv1 = dcll.ContinuousConv2D(
    in_channels=INPUT_DIM,
    out_channels=32,
    kernel_size=1,
    stride=1,
    padding=0,
    dilation=1,
    groups=1,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
).to(dev)

fc1 = CLLDenseModule(
    in_channels=32 * MAX_CONSTITUENTS,
    out_channels=64,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
).to(dev)

fc2 = CLLDenseModule(
    in_channels=64,
    out_channels=NUM_CLASSES,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
).to(dev)

# ======================
# FORCE DCLL alpha/tau to simple HLS-like one-step behavior
# eps0 = input
# eps1 = input
# pvmem = weight @ input
# spike = pvmem > 0
# ======================

conv1.alpha = torch.tensor([0.0], device=dev)
conv1.tau_m__dt = torch.tensor([1.0], device=dev)
conv1.alphas = torch.tensor([0.0], device=dev)
conv1.tau_s__dt = torch.tensor([1.0], device=dev)

fc1.alpha = torch.nn.Parameter(torch.tensor([0.0], device=dev), requires_grad=False)
fc1.tau_m__dt = torch.nn.Parameter(torch.tensor([1.0], device=dev), requires_grad=False)
fc1.alphas = torch.nn.Parameter(torch.tensor([0.0], device=dev), requires_grad=False)
fc1.tau_s__dt = torch.nn.Parameter(torch.tensor([1.0], device=dev), requires_grad=False)

fc2.alpha = torch.nn.Parameter(torch.tensor([0.0], device=dev), requires_grad=False)
fc2.tau_m__dt = torch.nn.Parameter(torch.tensor([1.0], device=dev), requires_grad=False)
fc2.alphas = torch.nn.Parameter(torch.tensor([0.0], device=dev), requires_grad=False)
fc2.tau_s__dt = torch.nn.Parameter(torch.tensor([1.0], device=dev), requires_grad=False)

# ======================
# RANDOM BINARY INPUT
# shape: B, C, H, W = 1, 3, 1, 10
# ======================

x = torch.randint(
    0, 2,
    (BATCH, INPUT_DIM, 1, MAX_CONSTITUENTS),
    dtype=torch.float32,
    device=dev
)

# ======================
# INTEGER WEIGHTS
# ======================

with torch.no_grad():
    conv1.weight.copy_(
        torch.randint(-2, 3, conv1.weight.shape, device=dev).float()
    )

    fc1.weight.copy_(
        torch.randint(-2, 3, fc1.weight.shape, device=dev).float()
    )

    fc2.weight.copy_(
        torch.randint(-2, 3, fc2.weight.shape, device=dev).float()
    )

# ======================
# INIT DCLL STATES
# ======================

# conv output shape: B, 32, 1, 10
conv1.state = conv1.NeuronState(
    eps0=torch.zeros((BATCH, 32, 1, MAX_CONSTITUENTS), device=dev),
    eps1=torch.zeros((BATCH, 32, 1, MAX_CONSTITUENTS), device=dev)
)

fc1.init_state(batch_size=BATCH, init_value=0)
fc2.init_state(batch_size=BATCH, init_value=0)

# ======================
# RUN DCLL FORWARD
# ======================

print("\n===== INPUT =====")
print(x.int().cpu().tolist())

# ---------- CONV1 ----------
spk1, pv1, pvmem1 = conv1(x, save=False)

print("\n===== CONV1 WEIGHT =====")
print(conv1.weight[:, :, 0, 0].int().cpu().tolist())

print("\n===== CONV1 PVMEM =====")
print(pvmem1.cpu().tolist())

print("\n===== CONV1 SPIKE =====")
print(spk1.int().cpu().tolist())

# ---------- FLATTEN ----------
flat = spk1.view(BATCH, -1)

print("\n===== FLATTEN =====")
print(flat.int().cpu().tolist())

# ---------- FC1 ----------
spk2, pv2, pvmem2 = fc1(flat)

print("\n===== FC1 WEIGHT (first 4 rows, first 16 inputs) =====")
print(fc1.weight[:4, :16].int().cpu().tolist())

print("\n===== FC1 PVMEM =====")
print(pvmem2.cpu().tolist())

print("\n===== FC1 SPIKE =====")
print(spk2.int().cpu().tolist())

# ---------- FC2 ----------
spk3, pv3, pvmem3 = fc2(spk2.float())

print("\n===== FC2 WEIGHT =====")
print(fc2.weight.int().cpu().tolist())

print("\n===== FC2 PVMEM =====")
print(pvmem3.cpu().tolist())

print("\n===== FINAL SPIKE OUTPUT =====")
print(spk3.int().cpu().tolist())

device: cuda
Sigmoid()
Continuous 2D

===== INPUT =====
[[[[1, 1, 1, 0, 1, 0, 1, 1, 0, 1]], [[0, 1, 0, 0, 0, 0, 0, 1, 0, 1]], [[0, 1, 1, 1, 0, 1, 0, 0, 1, 1]]]]

===== CONV1 WEIGHT =====
[[2, -2, 1], [2, -2, -2], [-1, -1, -2], [2, 1, 2], [2, -2, 0], [-1, -1, -2], [1, -2, -2], [0, 1, 2], [1, -1, -1], [1, 1, 1], [1, -2, -1], [2, -2, 0], [1, 2, -1], [-2, 2, 1], [-2, -2, 0], [2, 1, -1], [2, 2, 1], [2, -1, -2], [2, 2, 2], [2, -2, -1], [2, 0, -1], [-2, 1, 0], [0, -1, 1], [0, -1, 0], [-1, -2, 2], [-1, 0, 1], [2, -2, 0], [1, -2, 1], [2, 0, 2], [2, -1, 1], [-2, -1, -2], [-1, -2, 1]]

===== CONV1 PVMEM =====
[[[[2.0, 1.0, 3.0, 1.0, 2.0, 1.0, 2.0, 0.0, 1.0, 1.0]], [[2.0, -2.0, 0.0, -2.0, 2.0, -2.0, 2.0, 0.0, -2.0, -2.0]], [[-1.0, -4.0, -3.0, -2.0, -1.0, -2.0, -1.0, -2.0, -2.0, -4.0]], [[2.0, 5.0, 4.0, 2.0, 2.0, 2.0, 2.0, 3.0, 2.0, 5.0]], [[2.0, 0.0, 2.0, 0.0, 2.0, 0.0, 2.0, 0.0, 0.0, 0.0]], [[-1.0, -4.0, -3.0, -2.0, -1.0, -2.0, -1.0, -2.0, -2.0, -4.0]], [[1.0, -3.0, -1.0, -2.0, 1.0, -2.0, 1.0, -1

In [8]:
import torch
import numpy as np
import pytorch_libdcll as dcll
from pytorch_libdcll import CLLDenseModule

# ======================
# CONFIG
# ======================
BATCH = 1
INPUT_DIM = 3
MAX_CONSTITUENTS = 10
NUM_CLASSES = 5

torch.manual_seed(0)
np.random.seed(0)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

# ======================
# DCLL MODEL: square Conv2D
# input: 1 x 3 x 10 x 10
# conv1: 3 -> 32, 1x1
# fc1: 3200 -> 64
# fc2: 64 -> 5
# ======================

conv1 = dcll.ContinuousConv2D(
    in_channels=INPUT_DIM,
    out_channels=32,
    kernel_size=1,
    stride=1,
    padding=0,
    dilation=1,
    groups=1,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
).to(dev)

fc1 = CLLDenseModule(
    in_channels=32 * MAX_CONSTITUENTS * MAX_CONSTITUENTS,
    out_channels=64,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
).to(dev)

fc2 = CLLDenseModule(
    in_channels=64,
    out_channels=NUM_CLASSES,
    bias=False,
    alpha=0.0,
    alphas=0.0,
    spiking=True
).to(dev)

# ======================
# FORCE one-step HLS-like DCLL behavior
# ======================

conv1.alpha = torch.tensor([0.0], device=dev)
conv1.tau_m__dt = torch.tensor([1.0], device=dev)
conv1.alphas = torch.tensor([0.0], device=dev)
conv1.tau_s__dt = torch.tensor([1.0], device=dev)

for layer in [fc1, fc2]:
    layer.alpha = torch.nn.Parameter(torch.tensor([0.0], device=dev), requires_grad=False)
    layer.tau_m__dt = torch.nn.Parameter(torch.tensor([1.0], device=dev), requires_grad=False)
    layer.alphas = torch.nn.Parameter(torch.tensor([0.0], device=dev), requires_grad=False)
    layer.tau_s__dt = torch.nn.Parameter(torch.tensor([1.0], device=dev), requires_grad=False)

# ======================
# RANDOM SQUARE INPUT
# shape: B, C, H, W = 1, 3, 10, 10
# ======================

x = torch.randint(
    0, 2,
    (BATCH, INPUT_DIM, MAX_CONSTITUENTS, MAX_CONSTITUENTS),
    dtype=torch.float32,
    device=dev
)

# ======================
# INTEGER WEIGHTS
# ======================

with torch.no_grad():
    conv1.weight.copy_(
        torch.randint(-2, 3, conv1.weight.shape, device=dev).float()
    )

    fc1.weight.copy_(
        torch.randint(-2, 3, fc1.weight.shape, device=dev).float()
    )

    fc2.weight.copy_(
        torch.randint(-2, 3, fc2.weight.shape, device=dev).float()
    )

# ======================
# INIT DCLL STATES
# ======================

conv1.state = conv1.NeuronState(
    eps0=torch.zeros((BATCH, 32, MAX_CONSTITUENTS, MAX_CONSTITUENTS), device=dev),
    eps1=torch.zeros((BATCH, 32, MAX_CONSTITUENTS, MAX_CONSTITUENTS), device=dev)
)

fc1.init_state(batch_size=BATCH, init_value=0)
fc2.init_state(batch_size=BATCH, init_value=0)

# ======================
# RUN DCLL FORWARD
# ======================

print("\n===== INPUT =====")
print(x.int().cpu().tolist())

# ---------- CONV1 ----------
spk1, pv1, pvmem1 = conv1(x, save=False)

print("\n===== CONV1 WEIGHT =====")
print(conv1.weight[:, :, 0, 0].int().cpu().tolist())

print("\n===== CONV1 PVMEM =====")
print(pvmem1.cpu().tolist())

print("\n===== CONV1 SPIKE =====")
print(spk1.int().cpu().tolist())

# ---------- FLATTEN ----------
flat = spk1.view(BATCH, -1)

print("\n===== FLATTEN SHAPE =====")
print(flat.shape)

print("\n===== FLATTEN =====")
print(flat.int().cpu().tolist())

# ---------- FC1 ----------
spk2, pv2, pvmem2 = fc1(flat)

print("\n===== FC1 WEIGHT (first 4 rows, first 16 inputs) =====")
print(fc1.weight[:4, :16].int().cpu().tolist())

print("\n===== FC1 PVMEM =====")
print(pvmem2.cpu().tolist())

print("\n===== FC1 SPIKE =====")
print(spk2.int().cpu().tolist())

# ---------- FC2 ----------
spk3, pv3, pvmem3 = fc2(spk2.float())

print("\n===== FC2 WEIGHT =====")
print(fc2.weight.int().cpu().tolist())

print("\n===== FC2 PVMEM =====")
print(pvmem3.cpu().tolist())

print("\n===== FINAL SPIKE OUTPUT =====")
print(spk3.int().cpu().tolist())

device: cuda
Sigmoid()
Continuous 2D

===== INPUT =====
[[[[1, 1, 1, 0, 1, 0, 1, 1, 0, 1], [0, 1, 0, 0, 0, 0, 0, 1, 0, 1], [0, 1, 1, 1, 0, 1, 0, 0, 1, 1], [0, 1, 0, 1, 0, 0, 0, 1, 0, 0], [0, 0, 1, 0, 0, 1, 1, 1, 0, 1], [1, 1, 1, 1, 0, 0, 1, 1, 1, 1], [1, 1, 1, 1, 1, 0, 1, 0, 1, 1], [1, 0, 0, 0, 1, 1, 0, 1, 0, 1], [1, 1, 1, 1, 1, 1, 1, 0, 1, 0], [0, 1, 0, 0, 1, 1, 0, 0, 0, 1]], [[0, 0, 1, 1, 0, 1, 0, 0, 1, 0], [1, 1, 1, 1, 0, 1, 0, 0, 0, 1], [1, 1, 0, 1, 1, 1, 1, 1, 1, 0], [1, 0, 1, 0, 0, 0, 0, 0, 0, 0], [1, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 0, 0, 1, 1, 1, 0, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 0], [0, 1, 1, 0, 0, 1, 1, 0, 0, 1], [1, 0, 1, 0, 0, 1, 0, 0, 1, 0], [0, 1, 1, 1, 0, 0, 0, 0, 1, 0]], [[1, 1, 1, 0, 0, 0, 1, 0, 1, 1], [1, 1, 0, 1, 1, 1, 0, 1, 1, 0], [0, 0, 1, 1, 0, 0, 0, 0, 1, 1], [1, 0, 1, 1, 0, 1, 1, 1, 1, 0], [0, 0, 0, 0, 1, 1, 0, 1, 1, 0], [0, 1, 1, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, 1], [1, 0, 1, 1, 0, 1, 1, 0, 0, 0], [0, 1, 1, 1, 0, 1, 0, 1, 1, 1], [0, 0, 0,

In [11]:
import numpy as np

def write_cpp_array_1d(f, name, arr, per_line=16):
    arr = np.array(arr).astype(int).reshape(-1)
    f.write(f"static const int {name}[{len(arr)}] = {{\n")
    for i in range(0, len(arr), per_line):
        line = ", ".join(map(str, arr[i:i+per_line]))
        if i + per_line < len(arr):
            f.write("  " + line + ",\n")
        else:
            f.write("  " + line + "\n")
    f.write("};\n\n")

def write_cpp_array_2d(f, name, arr):
    arr = np.array(arr).astype(int)
    rows, cols = arr.shape
    f.write(f"static const int {name}[{rows}][{cols}] = {{\n")
    for r in range(rows):
        line = ", ".join(map(str, arr[r]))
        if r < rows - 1:
            f.write("  {" + line + "},\n")
        else:
            f.write("  {" + line + "}\n")
    f.write("};\n\n")


# ======================
# AFTER RUNNING MODEL
# ======================
# Square Conv2D:
# x    shape = [1, 3, 10, 10]
# spk1 shape = [1, 32, 10, 10]

conv_in   = x.int().cpu().numpy().reshape(-1)                    # [300]
conv_w    = model.conv1.weight[:, :, 0, 0].int().cpu().numpy()    # [32][3]
conv_spk  = spk1.int().cpu().numpy().reshape(-1)                 # [3200]

fc1_in    = spk1.float().view(-1).int().cpu().numpy()             # [3200]
fc1_w     = model.fc1.weight.int().cpu().numpy()                  # [64][3200]
fc1_spk   = spk2.view(-1).int().cpu().numpy()                     # [64]

fc2_in    = spk2.float().view(-1).int().cpu().numpy()             # [64]
fc2_w     = model.fc2.weight.int().cpu().numpy()                  # [5][64]
fc2_spk   = spk3.view(-1).int().cpu().numpy()                     # [5]


with open("conv1full_ref.h", "w") as f:
    f.write("#ifndef CONV1FULL_REF_H\n#define CONV1FULL_REF_H\n\n")
    write_cpp_array_1d(f, "CONV1_IN", conv_in)
    write_cpp_array_2d(f, "CONV1_W", conv_w)
    write_cpp_array_1d(f, "CONV1_EXPECT_SPK", conv_spk)
    f.write("#endif\n")

with open("fc1full_ref.h", "w") as f:
    f.write("#ifndef FC1FULL_REF_H\n#define FC1FULL_REF_H\n\n")
    write_cpp_array_1d(f, "FC1_IN", fc1_in)
    write_cpp_array_2d(f, "FC1_W", fc1_w)
    write_cpp_array_1d(f, "FC1_EXPECT_SPK", fc1_spk)
    f.write("#endif\n")

with open("fc2full_ref.h", "w") as f:
    f.write("#ifndef FC2FULL_REF_H\n#define FC2FULL_REF_H\n\n")
    write_cpp_array_1d(f, "FC2_IN", fc2_in)
    write_cpp_array_2d(f, "FC2_W", fc2_w)
    write_cpp_array_1d(f, "FC2_EXPECT_SPK", fc2_spk)
    f.write("#endif\n")

In [12]:
import numpy as np

def write_cpp_array_1d(f, name, arr, per_line=16):
    arr = np.array(arr).astype(int).reshape(-1)
    f.write(f"static const int {name}[{len(arr)}] = {{\n")
    for i in range(0, len(arr), per_line):
        line = ", ".join(map(str, arr[i:i+per_line]))
        if i + per_line < len(arr):
            f.write("  " + line + ",\n")
        else:
            f.write("  " + line + "\n")
    f.write("};\n\n")

def write_cpp_array_2d(f, name, arr):
    arr = np.array(arr).astype(int)
    rows, cols = arr.shape
    f.write(f"static const int {name}[{rows}][{cols}] = {{\n")
    for r in range(rows):
        line = ", ".join(map(str, arr[r]))
        if r < rows - 1:
            f.write("  {" + line + "},\n")
        else:
            f.write("  {" + line + "}\n")
    f.write("};\n\n")


# ======================
# AFTER RUNNING DCLL:
# spk1, spk2, spk3 already exist
# x shape    = [1, 3, 10, 10]
# spk1 shape = [1, 32, 10, 10]
# ======================

# IMPORTANT:
# FINN ConvLayer_Batch stream order is pixel-major:
# input:  B, H, W, C_in
# output: B, H, W, C_out

conv_in = (
    x
    .permute(0, 2, 3, 1)     # B,C,H,W -> B,H,W,C
    .contiguous()
    .int()
    .cpu()
    .numpy()
    .reshape(-1)
)  # [300]

conv_w = (
    conv1.weight[:, :, 0, 0]
    .int()
    .cpu()
    .numpy()
)  # [32][3]

conv_spk = (
    spk1
    .permute(0, 2, 3, 1)     # B,OC,H,W -> B,H,W,OC
    .contiguous()
    .int()
    .cpu()
    .numpy()
    .reshape(-1)
)  # [3200]


# FC1 input must match Python flatten:
# PyTorch flatten is B, OC, H, W -> OC-major
fc1_in = (
    spk1
    .contiguous()
    .view(-1)
    .int()
    .cpu()
    .numpy()
)  # [3200]

fc1_w = fc1.weight.int().cpu().numpy()      # [64][3200]
fc1_spk = spk2.view(-1).int().cpu().numpy() # [64]

fc2_in = spk2.view(-1).int().cpu().numpy()  # [64]
fc2_w = fc2.weight.int().cpu().numpy()      # [5][64]
fc2_spk = spk3.view(-1).int().cpu().numpy() # [5]


with open("conv1full_ref.h", "w") as f:
    f.write("#ifndef CONV1FULL_REF_H\n#define CONV1FULL_REF_H\n\n")
    write_cpp_array_1d(f, "CONV1_IN", conv_in)
    write_cpp_array_2d(f, "CONV1_W", conv_w)
    write_cpp_array_1d(f, "CONV1_EXPECT_SPK", conv_spk)
    f.write("#endif\n")

with open("fc1full_ref.h", "w") as f:
    f.write("#ifndef FC1FULL_REF_H\n#define FC1FULL_REF_H\n\n")
    write_cpp_array_1d(f, "FC1_IN", fc1_in)
    write_cpp_array_2d(f, "FC1_W", fc1_w)
    write_cpp_array_1d(f, "FC1_EXPECT_SPK", fc1_spk)
    f.write("#endif\n")

with open("fc2full_ref.h", "w") as f:
    f.write("#ifndef FC2FULL_REF_H\n#define FC2FULL_REF_H\n\n")
    write_cpp_array_1d(f, "FC2_IN", fc2_in)
    write_cpp_array_2d(f, "FC2_W", fc2_w)
    write_cpp_array_1d(f, "FC2_EXPECT_SPK", fc2_spk)
    f.write("#endif\n")

print("Wrote conv1full_ref.h, fc1full_ref.h, fc2full_ref.h")

Wrote conv1full_ref.h, fc1full_ref.h, fc2full_ref.h
